In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram


# needed for running quantum programs on IBM backends
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [ ]:
# Number of qubits in the GHZ state
n = 5

# Create the quantum circuit
ghz_circuit = QuantumCircuit(n)

# Create an equal superposition on the first qubit
ghz_circuit.h(0)

# Entangle the remaining qubits with the first qubit
for qubit in range(1, n):
    ghz_circuit.cx(0, qubit)

# Measure all qubits
ghz_circuit.measure_all()

# Visualize the circuit
print(ghz_circuit)

In [ ]:
from qiskit_aer import AerSimulator
ideal_simulator = AerSimulator()

# Create a sampler that uses the ideal simulator
ideal_sampler = Sampler(mode=ideal_simulator)

In [ ]:
# Number of circuit executions
shots = 1000

# Run the GHZ circuit
ideal_job = ideal_sampler.run(
    [ghz_circuit],
    shots=shots
)

# Retrieve the results
ideal_result = ideal_job.result()
ideal_counts = ideal_result[0].data.meas.get_counts()

print(ideal_counts )

In [ ]:
# Plot, save, and display the ideal results

fig, ax = plt.subplots(figsize=(6, 4))

plot_histogram(
    ideal_counts,
    ax=ax,
    bar_labels=False
)

ax.set_title(
    "Ideal Simulator",
    fontsize=18,
    pad=12
)

ax.set_xlabel("Measured Bitstring", fontsize=15)
ax.set_ylabel("Count", fontsize=15)

# Use a consistent scale across ideal and hardware results
ax.set_ylim(0, 0.7 * shots)

ax.tick_params(
    axis="x",
    labelsize=13
)

ax.tick_params(
    axis="y",
    labelsize=13
)

fig.tight_layout()

fig.savefig(
    f"GHZ_{n}_ideal.png",
    dpi=300,
    bbox_inches="tight"
)


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="CICkNyjUE4CvyjcP-6c2bD1b-gEh4Rrm6oesp64tvHPv",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/0a9a0d0e6b5d495db9abd0d7e2ee37f9:7358d162-4ab0-4564-a807-275591f86f67::",
    set_as_default=True,
    overwrite=True
)

In [ ]:
# Import the IBM Quantum service
from qiskit_ibm_runtime import QiskitRuntimeService

# Connect using the saved account
service = QiskitRuntimeService()

print("Connected to IBM Quantum.")

In [ ]:
# Get the available operational quantum backends
from qiskit import transpile
available_backends = service.backends(
    simulator=False,
    operational=True
)

# Print the backend names
print('Available IBM Quantum computers')
for backend in available_backends:
    print(f'    {backend.name}: {backend.num_qubits}-qubit device')

In [ ]:
backend1 = service.backend('ibm_fez')

print(f"Selected backend: {backend1.name}")
print(f"Number of qubits: {backend1.num_qubits}")
print(f"Pending jobs: {backend1.status().pending_jobs}")

In [ ]:
backend2 = service.backend('ibm_marrakesh')

print(f"Selected backend: {backend2.name}")
print(f"Number of qubits: {backend2.num_qubits}")
print(f"Pending jobs: {backend2.status().pending_jobs}")

In [ ]:
def load_job_results(filename):
    job_id_filename = filename

    # !!! you cannot retreive my job, even if you have access to the job ID :) 

    with open(job_id_filename, "r") as file:
        ibm_job_id = file.read().strip()

    print(f"Loaded job ID: {ibm_job_id}")

    # Retrieve the job from IBM Quantum
    retrieved_job = service.job(ibm_job_id)

    # Retrieve the backend information from the job
    job_backend = retrieved_job.backend()

    print(f"Job ID: {retrieved_job.job_id()}")
    print(f"Backend: {job_backend.name}")
    print(f"Job status: {retrieved_job.status()}")

    ibm_result = retrieved_job.result()
    ibm_counts = ibm_result[0].data.meas.get_counts()

    # Get the name of the backend used for execution
    backend_name = retrieved_job.backend().name

    print(f"Backend: {backend_name}")
    print(ibm_counts)

    fig, ax = plt.subplots(figsize=(6, 4))

    plot_histogram(
        ibm_counts,
        ax=ax,
        number_to_keep=10,
        bar_labels=False
    )

    ax.set_title(
        backend_name,
        fontsize=18,
        pad=12
    )

    ax.set_xlabel("Measured Bitstring", fontsize=15)
    ax.set_ylabel("Count", fontsize=15)

    # Use the same scale as the ideal-results figure
    ax.set_ylim(0, 0.7 * shots)

    ax.tick_params(
        axis="x",
        labelsize=12,
        labelrotation=45
    )

    ax.tick_params(
        axis="y",
        labelsize=13
    )

    fig.tight_layout()

    fig.savefig(
        f"GHZ_{n}_{backend_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

In [ ]:
load_job_results("GHZ_5_<IBMBackend('ibm_fez')>_ibm_job_id.txt")
load_job_results("GHZ_5_<IBMBackend('ibm_marrakesh')>_ibm_job_id.txt")

In [ ]:
def gather_stats(backend):
    # Display general information about the selected backend
    target = backend.target

    print(f"Backend name: {backend.name}")
    print(f"Number of qubits: {backend.num_qubits}")
    print(f"Backend version: {backend.backend_version}")
    print(f"Maximum circuits per job: {backend.max_circuits}")

    backend_properties = backend.properties(refresh=True)

    print(f"Backend: {backend.name}")
    print(f"Properties updated: {backend_properties.last_update_date}")

    qubit_0_properties = backend_properties.qubit_property(0)

    for property_name, property_data in qubit_0_properties.items():
        value, measured_at = property_data

        print(f"{property_name}: {value}")

    initialization_errors = []
    initialization_timestamps = []

    for qubit in range(backend.num_qubits):
        qubit_properties = backend_properties.qubit_property(qubit)

        if "init_error" in qubit_properties:
            error, measured_at = qubit_properties["init_error"]

            initialization_errors.append(error)
            initialization_timestamps.append(measured_at)
        else:
            initialization_errors.append(np.nan)
            initialization_timestamps.append(None)


    # Print the initialization errors

    for qubit, error in enumerate(initialization_errors):
        if np.isnan(error):
            print(f"Qubit {qubit}: Not reported")
        else:
            print(f"Qubit {qubit}: {100 * error:.3f}%")

    # Retrieve readout characteristics for each physical qubit

    prob_1_given_0 = []
    prob_0_given_1 = []
    readout_errors = []
    readout_durations = []

    for qubit in range(backend.num_qubits):
        qubit_properties = backend_properties.qubit_property(qubit)

        # Asymmetric misclassification probabilities
        p_1_0 = qubit_properties.get(
            "prob_meas1_prep0",
            (np.nan, None)
        )[0]

        p_0_1 = qubit_properties.get(
            "prob_meas0_prep1",
            (np.nan, None)
        )[0]

        # Average readout error and measurement duration
        measurement_properties = backend.target["measure"][(qubit,)]

        prob_1_given_0.append(p_1_0)
        prob_0_given_1.append(p_0_1)
        readout_errors.append(measurement_properties.error)
        readout_durations.append(measurement_properties.duration)


    # Print the readout characteristics
    print("readout errors:")

    print(
        f"{'Qubit':>5} "
        f"{'P(1|0)':>10} "
        f"{'P(0|1)':>10} "
        f"{'Avg. Error':>12} "
        f"{'Duration (µs)':>15}"
    )
    for qubit in range(backend.num_qubits):
        print(
            f"{qubit:>5} "
            f"{100 * prob_1_given_0[qubit]:>9.3f}% "
            f"{100 * prob_0_given_1[qubit]:>9.3f}% "
            f"{100 * readout_errors[qubit]:>11.3f}% "
            f"{1e6 * readout_durations[qubit]:>14.3f}"
        )


    print("\n" + "=" * 70)
    print("LATEX CHART: P(1|0)")
    print("=" * 70)

    print(r"""
    \begin{figure}[H]
        \centering
        \begin{tikzpicture}
            \begin{axis}[
                width=\textwidth,
                height=8cm,
                ybar,
                bar width=2.5pt,
                xlabel={Qubit},
                ylabel={$P(1|0)$ Error (\%)},
                title={$P(1|0)$ Readout Error per Qubit},
                xmin=-1,
                xmax=""" + str(backend.num_qubits) + r""",
                ymin=0,
                ymax=65,
                xtick={0,10,20,30,40,50,60,70,80,90,100,110,120,130,140,150},
                grid=major,
                grid style={dashed,gray!30},
                enlarge x limits=false,
            ]

            \addplot[
                fill=blue!60,
                draw=blue!80
            ] coordinates {
    """)

    for qubit in range(backend.num_qubits):
        print(
            f"            ({qubit},{100 * prob_1_given_0[qubit]:.3f})"
        )

    print(r"""        };

        \end{axis}
        \end{tikzpicture}
        \caption{$P(1|0)$ readout error for each qubit.}
        \label{fig:readout_p10}
    \end{figure}
    """)


    print("\n" + "=" * 70)
    print("LATEX CHART: P(0|1)")
    print("=" * 70)

    print(r"""
    \begin{figure}[H]
        \centering
        \begin{tikzpicture}
            \begin{axis}[
                width=\textwidth,
                height=8cm,
                ybar,
                bar width=2.5pt,
                xlabel={Qubit},
                ylabel={$P(0|1)$ Error (\%)},
                title={$P(0|1)$ Readout Error per Qubit},
                xmin=-1,
                xmax=""" + str(backend.num_qubits) + r""",
                ymin=0,
                ymax=65,
                xtick={0,10,20,30,40,50,60,70,80,90,100,110,120,130,140,150},
                grid=major,
                grid style={dashed,gray!30},
                enlarge x limits=false,
            ]

            \addplot[
                fill=orange!70,
                draw=orange!90
            ] coordinates {
    """)

    for qubit in range(backend.num_qubits):
        print(
            f"            ({qubit},{100 * prob_0_given_1[qubit]:.3f})"
        )

    print(r"""        };

        \end{axis}
        \end{tikzpicture}
        \caption{$P(0|1)$ readout error for each qubit.}
        \label{fig:readout_p01}
    \end{figure}
    """)


    print("\n" + "=" * 70)
    print("LATEX CHART: AVERAGE READOUT ERROR")
    print("=" * 70)

    print(r"""
    \begin{figure}[H]
        \centering
        \begin{tikzpicture}
            \begin{axis}[
                width=\textwidth,
                height=8cm,
                ybar,
                bar width=2.5pt,
                xlabel={Qubit},
                ylabel={Average Readout Error (\%)},
                title={Average Readout Error per Qubit},
                xmin=-1,
                xmax=""" + str(backend.num_qubits) + r""",
                ymin=0,
                ymax=55,
                xtick={0,10,20,30,40,50,60,70,80,90,100,110,120,130,140,150},
                grid=major,
                grid style={dashed,gray!30},
                enlarge x limits=false,
            ]

            \addplot[
                fill=green!60,
                draw=green!80
            ] coordinates {
    """)

    for qubit in range(backend.num_qubits):
        print(
            f"            ({qubit},{100 * readout_errors[qubit]:.3f})"
        )

    print(r"""        };

        \end{axis}
        \end{tikzpicture}
        \caption{Average readout error for each qubit.}
        \label{fig:readout_average}
    \end{figure}
    """)


    # Retrieve T1 for each physical qubit
    t1_values = []

    for qubit in range(backend.num_qubits):
        qubit_properties = backend.qubit_properties(qubit)

        if qubit_properties is None or qubit_properties.t1 is None:
            t1_values.append(np.nan)
        else:
            # Convert seconds to microseconds
            t1_values.append(qubit_properties.t1 * 1e6)

    t1_values = np.array(t1_values)

    # Print T1 for every individual qubit
    print("\nLaTeX coordinates:")
    for qubit, t1 in enumerate(t1_values):
        if np.isfinite(t1):
            print(f"({qubit},{t1:.2f})")


    valid_t1 = np.isfinite(t1_values)

    shortest_t1_qubit = np.nanargmin(t1_values)
    longest_t1_qubit = np.nanargmax(t1_values)

    print(f"Average T1: {np.nanmean(t1_values):.2f} µs")
    print(
        f"Shortest T1: Qubit {shortest_t1_qubit}, "
        f"{t1_values[shortest_t1_qubit]:.2f} µs"
    )
    print(
        f"Longest T1: Qubit {longest_t1_qubit}, "
        f"{t1_values[longest_t1_qubit]:.2f} µs"
    )
    print("\nLaTeX summary:")
    print(f"Average & --- & {np.nanmean(t1_values):.2f} \\\\")
    print(f"Shortest & Qubit {shortest_t1_qubit} & "
        f"{t1_values[shortest_t1_qubit]:.2f} \\\\")
    print(f"Longest & Qubit {longest_t1_qubit} & "
        f"{t1_values[longest_t1_qubit]:.2f} \\\\")
    print(f"Median & --- & {np.nanmedian(t1_values):.2f} \\\\")




    t2_values = []

    for qubit in range(backend.num_qubits):
        qubit_properties = backend.qubit_properties(qubit)

        if qubit_properties is None or qubit_properties.t2 is None:
            t2_values.append(np.nan)
        else:
            # Convert seconds to microseconds
            t2_values.append(qubit_properties.t2 * 1e6)

    t2_values = np.array(t2_values)

    # Summarize the T2 values

    shortest_t2_qubit = np.nanargmin(t2_values)
    longest_t2_qubit = np.nanargmax(t2_values)

    print(f"Average T2: {np.nanmean(t2_values):.2f} µs")
    print(
        f"Shortest T2: Qubit {shortest_t2_qubit}, "
        f"{t2_values[shortest_t2_qubit]:.2f} µs"
    )
    print(
        f"Longest T2: Qubit {longest_t2_qubit}, "
        f"{t2_values[longest_t2_qubit]:.2f} µs"
    )

    print("\nLaTeX coordinates:")
    for qubit, t2 in enumerate(t2_values):
        if np.isfinite(t2):
            print(f"({qubit},{t2:.2f})")

    print("\nLaTeX summary:")
    print(f"Average & --- & {np.nanmean(t2_values):.2f} \\\\")
    print(f"Shortest & Qubit {shortest_t2_qubit} & "
        f"{t2_values[shortest_t2_qubit]:.2f} \\\\")
    print(f"Longest & Qubit {longest_t2_qubit} & "
        f"{t2_values[longest_t2_qubit]:.2f} \\\\")
    print(f"Median & --- & {np.nanmedian(t2_values):.2f} \\\\")


# Collect one- and two-qubit gate characteristics

    gate_records = []

    excluded_instructions = {
        "measure",
        "reset",
        "delay",
        "barrier"
    }

    for gate_name in sorted(target.operation_names):
        if gate_name in excluded_instructions:
            continue

        qubit_arguments = target.qargs_for_operation_name(gate_name)

        # Skip globally defined instructions without explicit qubit assignments
        if qubit_arguments is None:
            continue

        for qubits_used in qubit_arguments:
            if len(qubits_used) not in (1, 2):
                continue

            gate_properties = target[gate_name][qubits_used]

            if gate_properties is None:
                continue

            gate_records.append(
                {
                    "gate": gate_name,
                    "qubits": qubits_used,
                    "num_qubits": len(qubits_used),
                    "error": gate_properties.error,
                    "duration": gate_properties.duration
                }
            )
    one_qubit_gates = sorted({
        record["gate"]
        for record in gate_records
        if record["num_qubits"] == 1
    })

    two_qubit_gates = sorted({
        record["gate"]
        for record in gate_records
        if record["num_qubits"] == 2
    })

    print("One-qubit gates:", one_qubit_gates)
    print("Two-qubit gates:", two_qubit_gates)

    # Summarize error and duration by gate type

    print(
        f"{'Gate':>10} "
        f"{'Qubits':>8} "
        f"{'Instances':>10} "
        f"{'Median Error':>15} "
        f"{'Median Duration':>18}"
    )

    for gate_name in sorted({
        record["gate"]
        for record in gate_records
    }):
        records = [
            record
            for record in gate_records
            if record["gate"] == gate_name
        ]

        errors = [
            record["error"]
            for record in records
            if record["error"] is not None
        ]

        durations = [
            record["duration"]
            for record in records
            if record["duration"] is not None
        ]

        gate_size = records[0]["num_qubits"]

        median_error = (
            100 * np.median(errors)
            if errors else np.nan
        )

        median_duration = (
            1e9 * np.median(durations)
            if durations else np.nan
        )

        print(
            f"{gate_name:>10} "
            f"{gate_size:>8} "
            f"{len(records):>10} "
            f"{median_error:>14.4f}% "
            f"{median_duration:>15.2f} ns"
        )
    cz_records = []
    seen_links = set()

    if "cz" in target.operation_names:

        cz_qargs = target.qargs_for_operation_name("cz")

        if cz_qargs is not None:

            for qubits_used in cz_qargs:

                if len(qubits_used) != 2:
                    continue

                q0 = int(qubits_used[0])
                q1 = int(qubits_used[1])


                canonical_link = tuple(sorted((q0, q1)))

                if canonical_link in seen_links:
                    continue

                seen_links.add(canonical_link)

                gate_properties = target["cz"][qubits_used]

                if gate_properties is None:
                    continue

                error = gate_properties.error
                duration = gate_properties.duration

                if error is None:
                    error_percent = np.nan
                else:
                    error_percent = 100.0 * error

                if duration is None:
                    duration_ns = np.nan
                else:
                    duration_ns = 1e9 * duration

                cz_records.append(
                    {
                        "link": (q0, q1),
                        "q0": q0,
                        "q1": q1,
                        "error": error_percent,
                        "duration": duration_ns,
                    }
                )

    # Sort by physical link for reproducible output
    cz_records.sort(key=lambda record: (record["q0"], record["q1"]))

    print("\n" + "=" * 80)
    print("CZ GATE ERROR BY PHYSICAL QUBIT LINK")
    print("=" * 80)

    print(f"Number of CZ links: {len(cz_records)}")



    valid_cz_errors = [
        record["error"]
        for record in cz_records
        if np.isfinite(record["error"])
    ]

    if valid_cz_errors:
        print(
            f"Minimum CZ error: {np.min(valid_cz_errors):.4f}%"
        )
        print(
            f"Maximum CZ error: {np.max(valid_cz_errors):.4f}%"
        )
        print(
            f"Median CZ error:  {np.median(valid_cz_errors):.4f}%"
        )
        print(
            f"Mean CZ error:    {np.mean(valid_cz_errors):.4f}%"
        )

    print("\n" + "=" * 80)
    print("LATEX TABLE: CZ ERROR BY LINK")
    print("=" * 80)

    ncols = 3 

    n_records = len(cz_records)
    chunk_size = -(-n_records // ncols)  # ceiling division

    chunks = [
        cz_records[i:i + chunk_size]
        for i in range(0, n_records, chunk_size)
    ]

    max_len = max(len(chunk) for chunk in chunks) if chunks else 0
    for chunk in chunks:
        chunk.extend([None] * (max_len - len(chunk)))

    col_spec = " ".join(["r r r"] * len(chunks))
    header_cells = " & ".join(
        ["Qubit 0 & Qubit 1 & CZ Error (\\%)"] * len(chunks)
    )

    print(r"""
\begin{table}[H]
    \centering
    \begin{tcolorbox}[colback=white, colframe=black,
        boxrule=0.6pt, arc=1pt, left=4pt, right=4pt, top=4pt, bottom=4pt]
    \scriptsize
    \setlength{\tabcolsep}{4pt}
    \begin{tabular}{""" + col_spec + r"""}
        \toprule
        """ + header_cells + r""" \\
        \midrule""")

    for row_idx in range(max_len):

        row_cells = []

        for chunk in chunks:
            record = chunk[row_idx]

            if record is None:
                row_cells.append(" & & ")
                continue

            q0 = record["q0"]
            q1 = record["q1"]
            error = record["error"]

            if np.isfinite(error):
                row_cells.append(f"{q0} & {q1} & {error:.4f}")
            else:
                row_cells.append(f"{q0} & {q1} & ---")

        print("        " + " & ".join(row_cells) + r" \\")

    print(r"""        \bottomrule
    \end{tabular}
    \end{tcolorbox}
    \caption{CZ gate error for each physical qubit link.}
    \label{tab:cz_gate_errors_by_link}
\end{table}
""")

In [ ]:
gather_stats(backend1)

In [ ]:
gather_stats(backend2)